# Supply chain forecasting + contract RAG (Cloudera AI demo)

This notebook mirrors the **Student Loan Risk Demo** narrative: warehouse tables → training job → deployed **model_api.predict** driving dashboards.

**Dense NSN** `9150-01-123-4567` (Turbine Oil): ARIMA + gradient boosting + LSTM.  
**Sparse NSN** `4820-00-111-2222` (Legacy Valve): gradient boosting on gap/supplier features.  
**RAG**: retrieve clauses from `contracts/CON-7781_Turbine_Oil_Supply_Agreement.pdf`, fuse with structured spike metrics.

In [ ]:
import os, sys, json
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, os.path.join(ROOT, "utils"))
os.chdir(ROOT)

from forecasting_pipeline import run_training, DENSE_DEMO_NSN, SPARSE_DEMO_NSN
from data_access import load_price_history
from contract_rag import build_index, retrieve, load_index, structured_price_context
from data_access import load_procurement_transactions

DATA_RAW = os.path.join(ROOT, "data", "raw")
MODELS = os.path.join(ROOT, "models")
PDF = os.path.join(ROOT, "contracts", "CON-7781_Turbine_Oil_Supply_Agreement.pdf")
print("Dense demo NSN:", DENSE_DEMO_NSN, "| Sparse:", SPARSE_DEMO_NSN)

In [ ]:
# Train / refresh artifacts (same step as Workbench Job)
summary = run_training(data_dir=DATA_RAW, models_dir=MODELS)
print(json.dumps(summary, indent=2))
if os.path.isfile(PDF):
    build_index(PDF, MODELS)

In [ ]:
import matplotlib.pyplot as plt

ph = load_price_history(DATA_RAW)
dense = ph[ph["nsn"] == DENSE_DEMO_NSN].sort_values("date")
sparse = ph[ph["nsn"] == SPARSE_DEMO_NSN].sort_values("date")
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
dense.plot(ax=axes[0], x="date", y="unit_price", title="Dense monthly pricing", legend=False)
sparse.plot(ax=axes[1], x="date", y="unit_price", title="Sparse multi-year purchases", legend=False)
plt.tight_layout()

In [ ]:
idx = load_index(MODELS)
tx = load_procurement_transactions(DATA_RAW)
ctx = structured_price_context(DENSE_DEMO_NSN, "CON-7781", ph, tx, spike_month="2025-03-01")
hits = retrieve("energy surcharge lubricant index", idx, top_k=2)
print("Structured:", ctx)
print("Top clause score:", hits[0].score if hits else None)

## Deployed model parity

Call the same logic via **`model_api.predict`** (after `create_model.py`): `forecast_dense`, `forecast_sparse`, `explain_spike`.